# SARA & Standard RAG Baseline Checkpoint Audit & Establishment Notebook

This notebook performs a **strict baseline-checkpoint audit and establishment phase** for the **QCCA (Query-Conditioned Evidence Allocation)** research project.

### Core Tasks:
1. **System Resource & Hardware Profiling**: Measure System RAM, GPU VRAM, PyTorch/CUDA environment.
2. **Training Configuration Audit**: Compare  vs. .
3. **Checkpoint Discovery**: Audit filesystem for existing  and  weights.
4. **Data Prerequisites**: Construct document-level disjoint QASPER dataset splits.
5. **Model Loading & Resource Tracking**: Load checkpoints with peak VRAM/RAM metrics and run numerical sanity tests.

In [1]:
import os
import sys
import time
import json
import psutil
import torch
import pandas as pd
from pathlib import Path

# Ensure repository root is in python path
REPO_ROOT = Path(".." ).resolve()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

def print_system_metrics():
    ram = psutil.virtual_memory()
    print("=" * 60)
    print("SYSTEM RESOURCE & METRICS AUDIT")
    print("=" * 60)
    print(f"CPU RAM Total     : {ram.total / (1024**3):.2f} GB")
    print(f"CPU RAM Available : {ram.available / (1024**3):.2f} GB")
    print(f"CPU RAM Used      : {ram.used / (1024**3):.2f} GB ({ram.percent}%)")
    
    if torch.cuda.is_available():
        print(f"GPU Device        : {torch.cuda.get_device_name(0)}")
        total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        alloc_vram = torch.cuda.memory_allocated(0) / (1024**3)
        res_vram = torch.cuda.memory_reserved(0) / (1024**3)
        peak_vram = torch.cuda.max_memory_allocated(0) / (1024**3)
        print(f"GPU VRAM Total    : {total_vram:.2f} GB")
        print(f"GPU VRAM Allocated: {alloc_vram:.2f} GB")
        print(f"GPU VRAM Reserved : {res_vram:.2f} GB")
        print(f"GPU VRAM Peak     : {peak_vram:.2f} GB")
    else:
        print("GPU VRAM          : CUDA Not Available")
    print("=" * 60)

print_system_metrics()

SYSTEM RESOURCE & METRICS AUDIT
CPU RAM Total     : 62.47 GB
CPU RAM Available : 58.82 GB
CPU RAM Used      : 3.66 GB (5.9%)
GPU VRAM          : CUDA Not Available


## 1. Official Training Configurations Audit Matrix

In [2]:
import yaml

rag_cfg_path = REPO_ROOT / "config/language_modeling/Finetune_RAG_qasper_pubbase.yaml"
sara_cfg_path = REPO_ROOT / "config/language_modeling/Finetune_SARA_qasper_pubbase.yaml"

with open(rag_cfg_path) as f:
    rag_cfg = yaml.safe_load(f)

with open(sara_cfg_path) as f:
    sara_cfg = yaml.safe_load(f)

audit_params = [
    ("Base Model", "model_name_or_path"),
    ("Compressor", "compressor_name_or_path"),
    ("Training Dataset", "train_file"),
    ("Dev File", "dev_file"),
    ("Max Train Samples", "max_train_samples"),
    ("Epochs", "num_train_epochs"),
    ("Max Train Steps", "max_train_steps"),
    ("Per Device Batch Size", "per_device_train_batch_size"),
    ("Grad Accumulation Steps", "gradient_accumulation_steps"),
    ("LM LoRA LR", "model_learning_rate"),
    ("Projector LR", "projector_learning_rate"),
    ("LR Scheduler", "lr_scheduler_type"),
    ("Warmup Ratio", "warmup_ratio"),
    ("Weight Decay", "weight_decay"),
    ("Seed", "seed"),
    ("Freeze Projector", "freeze_projector"),
    ("Text Evidence (num_evidence)", "num_evidence"),
    ("Compressed Evidence (num_add_evidence)", "num_additional_evidence"),
    ("Output Model Name", "output_model_name")
]

rows = []
for label, key in audit_params:
    rows.append({
        "Hyperparameter": label,
        "Standard RAG Config": str(rag_cfg.get(key)),
        "SARA Config": str(sara_cfg.get(key))
    })

df_audit = pd.DataFrame(rows)
display(df_audit)


,Hyperparameter,Standard RAG Config,SARA Config
0,Base Model,mistralai/Mistral-7B-Instruct-v0.2,mistralai/Mistral-7B-Instruct-v0.2
1,Compressor,None,Salesforce/SFR-Embedding-Mistral
2,Training Dataset,['data/reformatted/QASPER_train_split.jsonl'],['data/reformatted/QASPER_train_split.jsonl']
3,Dev File,None,None
4,Max Train Samples,2000,2000
5,Epochs,3,3
6,Max Train Steps,250,250
7,Per Device Batch Size,8,8
8,Grad Accumulation Steps,1,1
9,LM LoRA LR,2e-05,2e-05


## 2. Checkpoint Discovery & Verification

In [3]:
rag_ckpt_dir = REPO_ROOT / "checkpoints/finetune/rag_qasper_lora_r16_seed42"
sara_ckpt_dir = REPO_ROOT / "checkpoints/finetune/sara_qasper_proj_lr5e4_seed42"

rag_exists = (rag_ckpt_dir / "adapter_model.safetensors").exists() or (rag_ckpt_dir / "adapter_model.bin").exists()
sara_exists = (sara_ckpt_dir / "adapter_model.safetensors").exists() and (sara_ckpt_dir / "sara_extras.safetensors").exists()

print(f"RAG Adapter Exists  : {rag_exists} ({rag_ckpt_dir})")
print(f"SARA Adapter Exists : {sara_exists} ({sara_ckpt_dir})")

if not (rag_exists and sara_exists):
    print("\nRESULT: The baseline must be trained before evaluation.")
else:
    print("\nRESULT: Usable trained baseline checkpoints are present.")


RAG Adapter Exists  : False (/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/checkpoints/finetune/rag_qasper_lora_r16_seed42)
SARA Adapter Exists : False (/home/gunavenkat/Downloads/CS787-RAG-Project/Baselines/SARA-main/checkpoints/finetune/sara_qasper_proj_lr5e4_seed42)

RESULT: The baseline must be trained before evaluation.


## 3. Data Prerequisites & Split Preparation

In [4]:
data_split_manifest = REPO_ROOT / "data/manifests/qasper_split.json"
if data_split_manifest.exists():
    with open(data_split_manifest) as f:
        manifest = json.load(f)
    print("QASPER Document Split Manifest Found:")
    print(f"  Train Papers : {manifest.get('n_papers_train')} ({manifest.get('n_rows_train')} rows)")
    print(f"  Dev Papers   : {manifest.get('n_papers_dev')} ({manifest.get('n_rows_dev')} rows)")
    print(f"  Test Rows    : {manifest.get('n_rows_test')}")
else:
    print("Manifest not found. Run dataset construction pipeline.")


Manifest not found. Run dataset construction pipeline.


## 4. Official Training Trigger (Run if Checkpoints Absent)

Execute standard training commands when checkpoints are absent:


## 5. Checkpoint Loader & Memory Profiler Sanity Check

In [5]:
from src.model.loader import load_sara_for_eval
from src.eval.qa_models import HuggingFaceQAModel

def verify_and_profile_checkpoint(checkpoint_dir):
    if not os.path.exists(checkpoint_dir):
        print(f"Checkpoint dir {checkpoint_dir} does not exist. Train first.")
        return
    
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    ram_before = psutil.virtual_memory().used / (1024**3)
    
    print(f"Loading checkpoint from {checkpoint_dir}...")
    model, tokenizer = load_sara_for_eval(checkpoint_dir, device="cuda")
    
    load_time = time.time() - t0
    ram_after = psutil.virtual_memory().used / (1024**3)
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    
    print("=" * 60)
    print(f"CHECKPOINT LOAD & PROFILING SUMMARY ({checkpoint_dir})")
    print("=" * 60)
    print(f"Load Runtime       : {load_time:.2f} seconds")
    print(f"RAM Delta          : {ram_after - ram_before:.2f} GB")
    print(f"Peak GPU VRAM      : {peak_vram:.2f} GB")
    print(f"Compress Token ID  : {model.compress_token_id}")
    print(f"Tokenizer Vocab    : {len(tokenizer)}")
    
    if hasattr(model.get_base_model(), "projector"):
        proj_norm = sum(p.norm().item() for p in model.get_base_model().projector.parameters())
        print(f"Projector Param Norm: {proj_norm:.4f} (Non-zero check: OK)")
    else:
        print("Projector          : None (Projector-free RAG adapter)")
    print("=" * 60)

# Example usage once checkpoint directory is populated:
# verify_and_profile_checkpoint(str(sara_ckpt_dir))


/home/gunavenkat/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
